RFM INPUT PREPARATION 

In [1]:
import pandas as pd
from pathlib import Path 

In [2]:
# Load cleaned transaction data 

data_path = Path(r"C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\data\online_retail_cleaned.csv")

rfm_columns = [
    "Invoice", 
    "InvoiceDate", 
    "Customer ID", 
    "Total Amount"
]

df_rfm = pd.read_csv(
    data_path, usecols = rfm_columns, 
    parse_dates = ["InvoiceDate"]) 

print("RFM Input shape : ", df_rfm.shape)
print("\nColumns : ", df_rfm.columns.tolist())

RFM Input shape :  (778735, 4)

Columns :  ['Invoice', 'InvoiceDate', 'Customer ID', 'Total Amount']


In [3]:
# Define analysis date
# we define the analysis date as one day after the final observed transaction

analysis_date = df_rfm["InvoiceDate"].max() + pd.Timedelta(days = 1)

print("Last Transaction : ", df_rfm["InvoiceDate"].max())
print("Analysis Date : ", analysis_date)

Last Transaction :  2011-12-09 12:50:00
Analysis Date :  2011-12-10 12:50:00


In [4]:
# Create Customer-level RFM Dataset 

rfm = (
    df_rfm.groupby("Customer ID")
    .agg(Recency = ("InvoiceDate", lambda x : (analysis_date - x.max()).days), 
         Frequency = ("Invoice", "nunique"), 
         Monetary = ("Total Amount", "sum")).reset_index()
)

rfm["Monetary"] = rfm["Monetary"].round(2)

print("RFM Shape : ", rfm.shape)

print("\nRFM Preview : ")
print(rfm.head(10))

RFM Shape :  (5863, 4)

RFM Preview : 
   Customer ID  Recency  Frequency  Monetary
0      12346.0      326          4  77353.96
1      12347.0        2          8   4921.53
2      12348.0       75          5   2019.40
3      12349.0       19          4   4428.69
4      12350.0      310          1    334.40
5      12351.0      375          1    300.93
6      12352.0       36          9   2009.54
7      12353.0      204          2    406.76
8      12354.0      232          1   1079.40
9      12355.0      214          2    947.61


In [5]:
# RFM Validation 

print("\nRFM Validation")

print("\nUnique Customers : ", rfm["Customer ID"].nunique())
print("Missing Values : ", rfm.isna().sum().sum())
print("Duplicate Customers : ", rfm["Customer ID"].duplicated().sum())

print("\nMinimum Values : ")
print(rfm[["Recency", "Frequency", "Monetary"]].min())

print("\nMaximum Values : ")
print(rfm[["Recency", "Frequency", "Monetary"]].max())

# Assertions to prevent invalid data moving forward 
assert rfm["Customer ID"].is_unique 
assert rfm[["Recency", "Frequency", "Monetary"]].notna().all().all() 
assert rfm["Recency"].ge(1).all() 
assert rfm["Frequency"].ge(1).all() 
assert rfm["Monetary"].gt(0).all() 

print("\nRFM validation passed successfully.")


RFM Validation

Unique Customers :  5863
Missing Values :  0
Duplicate Customers :  0

Minimum Values : 
Recency      1.00
Frequency    1.00
Monetary     2.95
dtype: float64

Maximum Values : 
Recency         739.00
Frequency       379.00
Monetary     580987.04
dtype: float64

RFM validation passed successfully.


In [6]:
# RFM Descriptive summary 

rfm_summary = rfm[["Recency", "Frequency", "Monetary"]].describe() 

print("RFM Descriptive summary : ", "\n", rfm_summary)

RFM Descriptive summary :  
            Recency    Frequency       Monetary
count  5863.000000  5863.000000    5863.000000
mean    200.911308     6.273580    2937.540778
std     209.167897    12.846754   14367.008468
min       1.000000     1.000000       2.950000
25%      26.000000     1.000000     341.800000
50%      95.000000     3.000000     865.200000
75%     379.000000     7.000000    2245.980000
max     739.000000   379.000000  580987.040000


In [14]:
# Save RFM base dataset 

output_path = Path(r"C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\outputs")

output_path.mkdir(parents = True, exist_ok = True)

rfm_file = output_path / "rfm_base.csv"

rfm.to_csv(rfm_file, index = False) 

print("\nRFM base dataset saved to : ", "\n", rfm_file) 


RFM base dataset saved to :  
 C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\outputs\rfm_base.csv


RFM SCORING

In [7]:
# Create RFM Scores

# Recency 
# lower number of days = better customer engagement 
# Therefore, scoring is reversed : 5 = most recent 

rfm["R Score"] = pd.qcut( 
    rfm["Recency"].rank(method = "first"), 
    q = 5, labels = [5, 4, 3, 2, 1]
).astype(int) 

# Frequency 
# higher number of orders = better 

rfm["F Score"] = pd.qcut( 
    rfm["Frequency"].rank(method = "first"), 
    q = 5, labels = [1, 2, 3, 4, 5]
).astype(int) 

# Monetary 
# higher customer revenue = better 

rfm["M Score"] = pd.qcut( 
    rfm["Monetary"].rank(method = "first"), 
    q = 5, labels = [1, 2, 3, 4, 5]
).astype(int) 

In [8]:
# Create combined RFM Scores 

rfm["RFM Score"] = (
    rfm["R Score"].astype(str) + 
    rfm["F Score"].astype(str) + 
    rfm["M Score"].astype(str) 
)

rfm["RFM Total"] = ( 
    rfm["R Score"] + rfm["F Score"] + rfm["M Score"] 
)

In [9]:
# Score Validation 

print("\n--- RFM Score Validation ---")

print("\nR Score Distribution : ")
print(rfm["R Score"].value_counts().sort_index())

print("\nF Score Distribution : ")
print(rfm["F Score"].value_counts().sort_index())

print("\nM Score Distribution : ")
print(rfm["M Score"].value_counts().sort_index())

print("RFM Total Range : ")
print("Minimum : ", rfm["RFM Total"].min())
print("Maximum : ", rfm["RFM Total"].max())

print("\nMissing Scores : ")
print(rfm[["R Score", "F Score", "M Score", 
           "RFM Score", "RFM Total"]].isna().sum())


--- RFM Score Validation ---

R Score Distribution : 
R Score
1    1173
2    1172
3    1173
4    1172
5    1173
Name: count, dtype: int64

F Score Distribution : 
F Score
1    1173
2    1172
3    1173
4    1172
5    1173
Name: count, dtype: int64

M Score Distribution : 
M Score
1    1173
2    1172
3    1173
4    1172
5    1173
Name: count, dtype: int64
RFM Total Range : 
Minimum :  3
Maximum :  15

Missing Scores : 
R Score      0
F Score      0
M Score      0
RFM Score    0
RFM Total    0
dtype: int64


In [10]:
rfm[
    ["Customer ID", "Recency", "Frequency", "Monetary", 
    "R Score", "F Score", "M Score", "RFM Score", 
    "RFM Total"]
].sort_values("RFM Total", ascending = False).head(20) 

,Customer ID,Recency,Frequency,Monetary,R Score,F Score,M Score,RFM Score,RFM Total
3349,15738.0,19,41,13043.32,5,5,5,555,15
2369,14744.0,16,14,3660.08,5,5,5,555,15
2364,14739.0,4,25,11573.62,5,5,5,555,15
822,13174.0,16,12,7987.46,5,5,5,555,15
4896,17306.0,11,21,9782.35,5,5,5,555,15
4205,16607.0,18,25,5706.87,5,5,5,555,15
3213,15602.0,8,16,3137.96,5,5,5,555,15
5347,17769.0,13,17,7053.38,5,5,5,555,15
4687,17096.0,15,11,3697.93,5,5,5,555,15
3699,16094.0,12,11,3695.56,5,5,5,555,15


In [25]:
# Save scored RFM dataset 

rfm_scored_file = output_path / "rfm_scored.csv"

rfm.to_csv(rfm_scored_file, index = False)

print("nScored RFM Dataset saved to : ")
print(rfm_scored_file)

nScored RFM Dataset saved to : 
C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\outputs\rfm_scored.csv


RFM SEGMENTATION 

In [11]:
# Define customer segment 

def assign_segment(row) : 

    r = row["R Score"]
    f = row["F Score"]
    m = row["M Score"]

    # High-value, frequent and recently active 
    if r >= 4 and f >= 4 and m >= 4 : 
        return "VIP" 

    # Consistently active repeat customers 
    elif r >= 3 and f >= 4 and m >= 3 : 
        return "Steady Regular" 

    # Very recent customers with limited purchase history 
    elif r == 5 and f <= 2 and m <= 2 : 
        return "Recent Onboarding"

    # Recent customers with developing value / frequency 
    elif r >= 4 and f >= 2 and m >= 2 : 
        return " Growing Customer"  

    # Previously valuable / frequent customers showing inactivity 
    elif r <= 2 and f >= 3 and m >= 3 : 
        return "Customer at Risk" 

    # Low recent engagement and low purchasing activity 
    elif r <= 2 and f <= 2 and m <= 2 : 
        return "Inactive Customer" 

    # Recent but still developing customers 
    elif r >= 3 and f <= 2 and m <= 2 : 
        return "Promising Customer" 

    # Remaining mixed / irregular profiles 
    else : 
        return "Occasional Customer" 

rfm["Segment"] = rfm.apply(assign_segment, axis = 1)

print("Customer segmentation completed.")

Customer segmentation completed.


In [13]:
# Segmentation Validation 

print("\nSegment Distribution : ")
print(rfm["Segment"].value_counts().sort_values(ascending = False))

print("\nNumber of Segments : ", rfm["Segment"].nunique())

print("\nMissing Segment Values : ", rfm["Segment"].isna().sum())


Segment Distribution : 
Segment
VIP                    1282
Inactive Customer      1279
Occasional Customer     901
Steady Regular          654
Customer at Risk        622
 Growing Customer       594
Promising Customer      418
Recent Onboarding       113
Name: count, dtype: int64

Number of Segments :  8

Missing Segment Values :  0


In [17]:
# Segment Business Profile 

segment_profile = (rfm.groupby("Segment").agg(
    Customers = ("Customer ID", "count"), 
    Total_Revenue = ("Monetary", "sum"), 
    Avg_Revenue = ("Monetary", "mean"), 
    Avg_Recency = ("Recency", "mean"), 
    Avg_Frequency = ("Frequency", "mean")
).reset_index())

segment_profile["Revenue_Share_Pct"] = (
    segment_profile["Total_Revenue"] / segment_profile["Total_Revenue"].sum() * 100) 

segment_profile = segment_profile.sort_values(
    "Total_Revenue", 
    ascending=False
)

segment_profile["Total_Revenue"] = segment_profile["Total_Revenue"].round(2) 
segment_profile["Avg_Revenue"] = segment_profile["Avg_Revenue"].round(2) 
segment_profile["Avg_Recency"] = segment_profile["Avg_Recency"].round(2) 
segment_profile["Avg_Frequency"] = segment_profile["Avg_Frequency"].round(2) 
segment_profile["Revenue_Share_Pct"] = segment_profile["Revenue_Share_Pct"].round(2)  

print("\nSegment Business Profile : ")
print(segment_profile.to_string(index = False))


Segment Business Profile : 
            Segment  Customers  Total_Revenue  Avg_Revenue  Avg_Recency  Avg_Frequency  Revenue_Share_Pct
                VIP       1282    11730587.13      9150.22        19.66          17.12              68.11
     Steady Regular        654     1834406.87      2804.90        83.99           7.92              10.65
   Customer at Risk        622     1519929.10      2443.62       358.94           5.67               8.83
Occasional Customer        901      890104.70       987.91       251.15           2.51               5.17
   Growing Customer        594      780851.22      1314.56        25.96           2.86               4.53
  Inactive Customer       1279      322969.79       252.52       466.58           1.19               1.88
 Promising Customer        418      109564.79       262.12        83.33           1.20               0.64
  Recent Onboarding        113       34387.98       304.32        11.12           1.40               0.20


In [19]:
# Save final customer RFM segment 

rfm["Customer ID"] = rfm["Customer ID"].astype(int) 

output_path = (r"C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\outputs\customer_rfm_segments.csv") 

rfm.to_csv(output_path, index = False) 

print(f"\nFinal RFM Segmentation dataset saved to : \n{output_path}")
print(f"\nShape : {rfm.shape}")


Final RFM Segmentation dataset saved to : 
C:\Users\Admin\Downloads\Portfolio Projects\E-Commerce-RFM-Analytics\outputs\customer_rfm_segments.csv

Shape : (5863, 10)
